In [3]:
from __future__ import annotations
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

In [4]:
#===================CONFIG===================
RANDOM_STATE = 42
N_FOLDS = 5

TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.10
TEST_SIZE = 0.20

MIN_CATEGORY_HOLDOUT_SIZE = 100

In [5]:
INPUT_PATH = Path('/content/viclickbait_eda_features.csv')
OUTPUT_DIR = Path('/content/phase3')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Phase 3 config loaded')
print(f'Input path: {INPUT_PATH}')
print(f'Output dir: {OUTPUT_DIR}')

Phase 3 config loaded
Input path: /content/viclickbait_eda_features.csv
Output dir: /content/phase3


## 1. Load feature engineering

In [6]:
def load_phase2_features(input_path: Path = INPUT_PATH) -> pd.DataFrame:
    if not input_path.exists():
        raise FileNotFoundError(
            f'Không tìm thấy input Phase 2: {input_path}. '
        )
    df = pd.read_csv(input_path)
    print(f'Loaded input: {input_path}')
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    return df


df_raw = load_phase2_features()
display(df_raw.head(3))

Loaded input: /content/viclickbait_eda_features.csv
Shape: 3,414 rows x 38 columns


,id,title,lead_paragraph,source,category,publish_datetime_raw,publish_dt,label,label_str,title_char_len,...,title_has_quote,title_has_source_tag,lead_char_len,lead_word_len,lead_title_ratio,lead_n_cb_keywords,source_encoded,category_encoded,source_cb_rate_diagnostic,category_cb_rate_diagnostic
0,article_0001,Sân bay Vinh đóng cửa 6 tháng để nâng cấp,Nghệ AnCảng hàng không Vinh sẽ ngừng hoạt động...,VnExpress,Tin tức tổng hợp,2025-06-23T12:26:00+07:00,2025-06-23 05:26:00.000,0,Non-clickbait,41,...,0,0,155,34,3.690476,0,7,10,0.144737,0.040609
1,article_0002,5 người thoát nạn khi ôtô bị lũ cuốn,Tuyên QuangĐi theo Google Maps song lại được c...,VnExpress,Tin tức tổng hợp,2025-06-23T12:26:00+07:00,2025-06-23 05:26:00.000,0,Non-clickbait,36,...,0,0,142,31,3.837838,0,7,10,0.144737,0.040609
2,article_0004,Quy hoạch 9 khu chức năng phía đông TP HCM,9 đồ án quy hoạch 1/2000 được TP Thủ Đức phê d...,VnExpress,Tin tức tổng hợp,2025-06-23T11:56:00+07:00,2025-06-23 04:56:00.000,0,Non-clickbait,42,...,0,0,156,34,3.627907,0,7,10,0.144737,0.040609


## 2. Chuẩn hóa dữ liệu cho Split

Các bước xử lý:

- Kiểm tra cột bắt buộc.
- Chuẩn hóa `label` về 0/1.
- Tạo `label_str`.
- Tạo `duplicate_group` từ normalized title để audit leakage.
- Parse `publish_dt` nếu có.

In [7]:
def normalize_title(text: object) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(
        r'[^\w\sàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợ'
        r'ùúủũụưừứửữựỳýỷỹỵđ]',
        '',
        text,
    )
    return text.strip()

In [8]:
def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    required = ['id', 'title', 'label', 'source', 'category']
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns for Phase 3: {missing}')

    df = df.copy().reset_index(drop=True)

    if df['id'].duplicated().any():
        duplicated_ids = df.loc[df['id'].duplicated(keep=False), 'id'].unique()[:10]
        raise ValueError(f'Duplicated ids detected. Examples: {duplicated_ids}')

    if df['label'].dtype == object:
        label_norm = df['label'].astype(str).str.strip().str.lower()
        label_map = {
            'clickbait': 1,
            'non-clickbait': 0,
            'non clickbait': 0,
            '1': 1,
            '0': 0,
        }
        df['label'] = label_norm.map(label_map)

    if df['label'].isna().any():
        bad = df.loc[df['label'].isna(), ['id', 'label']].head()
        raise ValueError(f'Invalid labels detected:\n{bad}')

    df['label'] = df['label'].astype(int)
    df['label_str'] = df['label'].map({1: 'Clickbait', 0: 'Non-clickbait'})
    df['normalized_title'] = df['title'].apply(normalize_title)
    df['duplicate_group'] = df['normalized_title'].factorize()[0]

    if 'publish_dt' not in df.columns:
        if 'publish_datetime_raw' in df.columns:
            df['publish_dt'] = pd.to_datetime(df['publish_datetime_raw'], errors='coerce')
        elif 'publish_datetime' in df.columns:
            df['publish_dt'] = pd.to_datetime(df['publish_datetime'], errors='coerce')
        else:
            df['publish_dt'] = pd.NaT
    else:
        df['publish_dt'] = pd.to_datetime(df['publish_dt'], errors='coerce')

    return df

In [10]:
df = prepare_dataframe(df_raw)
print(f'Dataset sau chuẩn hóa: {df.shape[0]:,} rows x {df.shape[1]} columns')
display(df[['id', 'title', 'label', 'label_str', 'source', 'category', 'publish_dt', 'duplicate_group']].head())
display(df['label_str'].value_counts().to_frame('count'))

Dataset sau chuẩn hóa: 3,414 rows x 40 columns


,id,title,label,label_str,source,category,publish_dt,duplicate_group
0,article_0001,Sân bay Vinh đóng cửa 6 tháng để nâng cấp,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-23 05:26:00,0
1,article_0002,5 người thoát nạn khi ôtô bị lũ cuốn,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-23 05:26:00,1
2,article_0004,Quy hoạch 9 khu chức năng phía đông TP HCM,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-23 04:56:00,2
3,article_0006,Tổng Bí thư trao quyết định thành lập Cơ quan ...,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-23 04:03:00,3
4,article_0010,Quốc hội bước vào tuần làm việc cuối cùng của ...,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-22 22:00:00,4


,count
label_str,
Non-clickbait,2349
Clickbait,1065


In [11]:
def split_distribution(
    split_df: pd.DataFrame,
    split_name: str,
    dataset_total: int | None = None,
) -> pd.DataFrame:
    rows = []
    total = len(split_df) if dataset_total is None else dataset_total

    for split_value, part in split_df.groupby('split', dropna=False):
        n_total = len(part)
        n_clickbait = int(part['label'].sum())
        n_non_clickbait = int((part['label'] == 0).sum())
        rows.append({
            'protocol': split_name,
            'split': split_value,
            'n_total': n_total,
            'n_clickbait': n_clickbait,
            'n_non_clickbait': n_non_clickbait,
            'clickbait_rate': round(n_clickbait / n_total * 100, 2) if n_total else 0.0,
            'share_of_dataset': round(n_total / total * 100, 2) if total else 0.0,
        })

    return pd.DataFrame(rows)

## 3. Random stratified split 70/10/20

Split chính cho benchmark baseline.

In [12]:
def make_random_stratified_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_val, test = train_test_split(
        df,
        test_size=TEST_SIZE,
        stratify=df['label'],
        random_state=RANDOM_STATE,
    )

    validation_ratio_inside_train_val = VALIDATION_SIZE / (TRAIN_SIZE + VALIDATION_SIZE)
    train, validation = train_test_split(
        train_val,
        test_size=validation_ratio_inside_train_val,
        stratify=train_val['label'],
        random_state=RANDOM_STATE,
    )

    split_df = pd.concat([
        train.assign(split='train'),
        validation.assign(split='validation'),
        test.assign(split='test'),
    ], ignore_index=True)

    split_df = split_df[[
        'id', 'split', 'label', 'label_str', 'source', 'category',
        'publish_dt', 'duplicate_group'
    ]].sort_values(['split', 'id'])

    summary = split_distribution(split_df, 'random_stratified_70_10_20')
    return split_df, summary


random_split, random_summary = make_random_stratified_split(df)

display(random_summary)
display(random_split.head())

,protocol,split,n_total,n_clickbait,n_non_clickbait,clickbait_rate,share_of_dataset
0,random_stratified_70_10_20,test,683,213,470,31.19,20.01
1,random_stratified_70_10_20,train,2389,745,1644,31.18,69.98
2,random_stratified_70_10_20,validation,342,107,235,31.29,10.02


,id,split,label,label_str,source,category,publish_dt,duplicate_group
3076,article_0002,test,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-23 05:26:00,1
2995,article_0010,test,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-22 22:00:00,4
2739,article_0017,test,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-22 04:48:00,11
3122,article_0022,test,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-21 15:20:00,16
2781,article_0025,test,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2025-06-21 10:16:00,19


## 4. Stratified 5-fold split

Dùng để kiểm tra độ ổn định, ưu tiên cho Traditional ML và FastText.

In [13]:
def make_stratified_kfold(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_df = df[['id', 'label', 'label_str', 'source', 'category', 'duplicate_group']].copy()
    fold_df['fold'] = -1

    for fold, (_, validation_idx) in enumerate(skf.split(df, df['label'])):
        fold_df.loc[df.index[validation_idx], 'fold'] = fold

    if (fold_df['fold'] < 0).any():
        raise RuntimeError('Some rows were not assigned to a fold.')

    rows = []
    for fold in range(N_FOLDS):
        part = fold_df[fold_df['fold'] == fold].copy()
        part['split'] = f'fold_{fold}_validation'
        rows.append(
            split_distribution(
                part,
                f'stratified_kfold_{N_FOLDS}',
                dataset_total=len(fold_df),
            )
        )

    summary = pd.concat(rows, ignore_index=True)
    return fold_df.sort_values(['fold', 'id']), summary


kfold_split, kfold_summary = make_stratified_kfold(df)

display(kfold_summary)
display(kfold_split.head())

,protocol,split,n_total,n_clickbait,n_non_clickbait,clickbait_rate,share_of_dataset
0,stratified_kfold_5,fold_0_validation,683,213,470,31.19,20.01
1,stratified_kfold_5,fold_1_validation,683,213,470,31.19,20.01
2,stratified_kfold_5,fold_2_validation,683,213,470,31.19,20.01
3,stratified_kfold_5,fold_3_validation,683,213,470,31.19,20.01
4,stratified_kfold_5,fold_4_validation,682,213,469,31.23,19.98


,id,label,label_str,source,category,duplicate_group,fold
5,article_0011,0,Non-clickbait,VnExpress,Tin tức tổng hợp,5,0
7,article_0013,0,Non-clickbait,VnExpress,Tin tức tổng hợp,7,0
15,article_0021,0,Non-clickbait,VnExpress,Tin tức tổng hợp,15,0
17,article_0023,0,Non-clickbait,VnExpress,Tin tức tổng hợp,17,0
18,article_0024,0,Non-clickbait,VnExpress,Tin tức tổng hợp,18,0


## 5. Leave-one-source-out split

Dùng để kiểm tra domain generalization theo nguồn báo.

In [14]:
def make_leave_one_source_out(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    summary_rows = []

    for source in sorted(df['source'].dropna().unique()):
        part = df[['id', 'label', 'label_str', 'source', 'category', 'duplicate_group']].copy()
        part['heldout_source'] = source
        part['split'] = np.where(part['source'] == source, 'test', 'train')
        rows.append(part)

        summary = split_distribution(part, f'leave_one_source_out::{source}')
        summary['heldout_source'] = source
        summary_rows.append(summary)

    return pd.concat(rows, ignore_index=True), pd.concat(summary_rows, ignore_index=True)


source_split, source_summary = make_leave_one_source_out(df)

display(source_summary)
display(source_split.head())

,protocol,split,n_total,n_clickbait,n_non_clickbait,clickbait_rate,share_of_dataset,heldout_source
0,leave_one_source_out::24h,test,564,248,316,43.97,16.52,24h
1,leave_one_source_out::24h,train,2850,817,2033,28.67,83.48,24h
2,leave_one_source_out::Báo Mới,test,731,148,583,20.25,21.41,Báo Mới
3,leave_one_source_out::Báo Mới,train,2683,917,1766,34.18,78.59,Báo Mới
4,leave_one_source_out::Kênh14,test,52,40,12,76.92,1.52,Kênh14
5,leave_one_source_out::Kênh14,train,3362,1025,2337,30.49,98.48,Kênh14
6,leave_one_source_out::SaoStar,test,286,186,100,65.03,8.38,SaoStar
7,leave_one_source_out::SaoStar,train,3128,879,2249,28.10,91.62,SaoStar
8,leave_one_source_out::Thanh Niên,test,576,172,404,29.86,16.87,Thanh Niên
9,leave_one_source_out::Thanh Niên,train,2838,893,1945,31.47,83.13,Thanh Niên


,id,label,label_str,source,category,duplicate_group,heldout_source,split
0,article_0001,0,Non-clickbait,VnExpress,Tin tức tổng hợp,0,24h,train
1,article_0002,0,Non-clickbait,VnExpress,Tin tức tổng hợp,1,24h,train
2,article_0004,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2,24h,train
3,article_0006,0,Non-clickbait,VnExpress,Tin tức tổng hợp,3,24h,train
4,article_0010,0,Non-clickbait,VnExpress,Tin tức tổng hợp,4,24h,train


## 6. Category-held-out split

Chỉ giữ các category có ít nhất 100 mẫu để test held-out.

In [15]:
def make_category_heldout(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    category_counts = df['category'].value_counts()
    eligible_categories = sorted(
        category_counts[category_counts >= MIN_CATEGORY_HOLDOUT_SIZE].index.tolist()
    )

    rows = []
    summary_rows = []

    for category in eligible_categories:
        part = df[['id', 'label', 'label_str', 'source', 'category', 'duplicate_group']].copy()
        part['heldout_category'] = category
        part['split'] = np.where(part['category'] == category, 'test', 'train')
        rows.append(part)

        summary = split_distribution(part, f'category_heldout::{category}')
        summary['heldout_category'] = category
        summary_rows.append(summary)

    if not rows:
        empty = pd.DataFrame()
        return empty, empty

    return pd.concat(rows, ignore_index=True), pd.concat(summary_rows, ignore_index=True)


category_split, category_summary = make_category_heldout(df)

display(category_summary)
display(category_split.head())

,protocol,split,n_total,n_clickbait,n_non_clickbait,clickbait_rate,share_of_dataset,heldout_category
0,category_heldout::Chính trị & An ninh,test,173,29,144,16.76,5.07,Chính trị & An ninh
1,category_heldout::Chính trị & An ninh,train,3241,1036,2205,31.97,94.93,Chính trị & An ninh
2,category_heldout::Công nghệ & Khoa học,test,220,78,142,35.45,6.44,Công nghệ & Khoa học
3,category_heldout::Công nghệ & Khoa học,train,3194,987,2207,30.90,93.56,Công nghệ & Khoa học
4,category_heldout::Du lịch & Ẩm thực,test,109,47,62,43.12,3.19,Du lịch & Ẩm thực
5,category_heldout::Du lịch & Ẩm thực,train,3305,1018,2287,30.80,96.81,Du lịch & Ẩm thực
6,category_heldout::Giáo dục & Học đường,test,160,37,123,23.12,4.69,Giáo dục & Học đường
7,category_heldout::Giáo dục & Học đường,train,3254,1028,2226,31.59,95.31,Giáo dục & Học đường
8,category_heldout::Giải trí & Showbiz,test,238,168,70,70.59,6.97,Giải trí & Showbiz
9,category_heldout::Giải trí & Showbiz,train,3176,897,2279,28.24,93.03,Giải trí & Showbiz


,id,label,label_str,source,category,duplicate_group,heldout_category,split
0,article_0001,0,Non-clickbait,VnExpress,Tin tức tổng hợp,0,Chính trị & An ninh,train
1,article_0002,0,Non-clickbait,VnExpress,Tin tức tổng hợp,1,Chính trị & An ninh,train
2,article_0004,0,Non-clickbait,VnExpress,Tin tức tổng hợp,2,Chính trị & An ninh,train
3,article_0006,0,Non-clickbait,VnExpress,Tin tức tổng hợp,3,Chính trị & An ninh,train
4,article_0010,0,Non-clickbait,VnExpress,Tin tức tổng hợp,4,Chính trị & An ninh,train


## 7. Temporal chronological split

Split theo thời gian chỉ nên xem là exploratory nếu dữ liệu lệch mạnh về một năm.

In [16]:
def make_temporal_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    temporal_df = df[['id', 'label', 'label_str', 'source', 'category', 'publish_dt']].copy()
    temporal_df['split'] = 'excluded_missing_datetime'

    valid = temporal_df.dropna(subset=['publish_dt']).sort_values(['publish_dt', 'id']).copy()
    n_valid = len(valid)

    train_end = int(n_valid * TRAIN_SIZE)
    validation_end = int(n_valid * (TRAIN_SIZE + VALIDATION_SIZE))

    valid.loc[valid.index[:train_end], 'split'] = 'train'
    valid.loc[valid.index[train_end:validation_end], 'split'] = 'validation'
    valid.loc[valid.index[validation_end:], 'split'] = 'test'

    temporal_df.loc[valid.index, 'split'] = valid['split']
    temporal_df = temporal_df.sort_values(['split', 'publish_dt', 'id'])

    summary = split_distribution(temporal_df, 'temporal_chronological_70_10_20')

    temporal_note = {
        'n_total': int(len(temporal_df)),
        'n_valid_datetime': int(n_valid),
        'n_missing_datetime': int(temporal_df['publish_dt'].isna().sum()),
        'note': (
            'Temporal split is exploratory if most samples belong to a single year. '
            'Report this limitation explicitly.'
        ),
    }
    (OUTPUT_DIR / 'temporal_split_note.json').write_text(
        json.dumps(temporal_note, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

    return temporal_df, summary


temporal_split, temporal_summary = make_temporal_split(df)

display(temporal_summary)
display(temporal_split.head())

,protocol,split,n_total,n_clickbait,n_non_clickbait,clickbait_rate,share_of_dataset
0,temporal_chronological_70_10_20,excluded_missing_datetime,31,12,19,38.71,0.91
1,temporal_chronological_70_10_20,test,677,168,509,24.82,19.83
2,temporal_chronological_70_10_20,train,2368,799,1569,33.74,69.36
3,temporal_chronological_70_10_20,validation,338,86,252,25.44,9.90


,id,label,label_str,source,category,publish_dt,split
13,article_0019,0,Non-clickbait,VnExpress,Tin tức tổng hợp,NaT,excluded_missing_datetime
198,article_0246,0,Non-clickbait,Báo Mới,Thể thao,NaT,excluded_missing_datetime
199,article_0247,0,Non-clickbait,Báo Mới,Thể thao,NaT,excluded_missing_datetime
200,article_0248,0,Non-clickbait,Báo Mới,Thể thao,NaT,excluded_missing_datetime
201,article_0249,0,Non-clickbait,Báo Mới,Thể thao,NaT,excluded_missing_datetime


## 8. Metrics Config

In [17]:
def write_metrics_config() -> dict:
    metrics_config = {
        'primary_metric': 'macro_f1',
        'positive_label': 1,
        'positive_label_name': 'Clickbait',
        'required_metrics': [
            'macro_f1',
            'weighted_f1',
            'clickbait_precision',
            'clickbait_recall',
            'clickbait_f1',
            'balanced_accuracy',
            'confusion_matrix',
        ],
        'optional_metrics': ['roc_auc', 'pr_auc', 'mean_std_across_seeds'],
        'do_not_use_alone': ['accuracy'],
        'random_state': RANDOM_STATE,
    }

    (OUTPUT_DIR / 'metrics_config.json').write_text(
        json.dumps(metrics_config, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    return metrics_config


metrics_config = write_metrics_config()
metrics_config

{'primary_metric': 'macro_f1',
 'positive_label': 1,
 'positive_label_name': 'Clickbait',
 'required_metrics': ['macro_f1',
  'weighted_f1',
  'clickbait_precision',
  'clickbait_recall',
  'clickbait_f1',
  'balanced_accuracy',
  'confusion_matrix'],
 'optional_metrics': ['roc_auc', 'pr_auc', 'mean_std_across_seeds'],
 'do_not_use_alone': ['accuracy'],
 'random_state': 42}

In [18]:
# Export output
random_split.to_csv(OUTPUT_DIR / 'random_stratified_70_10_20.csv', index=False)
kfold_split.to_csv(OUTPUT_DIR / f'stratified_kfold_{N_FOLDS}.csv', index=False)
source_split.to_csv(OUTPUT_DIR / 'leave_one_source_out.csv', index=False)
category_split.to_csv(OUTPUT_DIR / 'category_heldout.csv', index=False)
temporal_split.to_csv(OUTPUT_DIR / 'temporal_chronological_70_10_20.csv', index=False)

split_summary = pd.concat([
    random_summary,
    kfold_summary,
    source_summary,
    category_summary,
    temporal_summary,
], ignore_index=True, sort=False)

split_summary.to_csv(OUTPUT_DIR / 'split_summary.csv', index=False)

print('Phase 3 split generation completed.')
print(f'Output directory: {OUTPUT_DIR}')
print('\nFiles written:')
for path in sorted(OUTPUT_DIR.glob('*')):
    print(f'  - {path}')

print('\nRandom split summary:')
display(random_summary)

Phase 3 split generation completed.
Output directory: /content/phase3

Files written:
  - /content/phase3/category_heldout.csv
  - /content/phase3/leave_one_source_out.csv
  - /content/phase3/metrics_config.json
  - /content/phase3/random_stratified_70_10_20.csv
  - /content/phase3/split_summary.csv
  - /content/phase3/stratified_kfold_5.csv
  - /content/phase3/temporal_chronological_70_10_20.csv
  - /content/phase3/temporal_split_note.json

Random split summary:


,protocol,split,n_total,n_clickbait,n_non_clickbait,clickbait_rate,share_of_dataset
0,random_stratified_70_10_20,test,683,213,470,31.19,20.01
1,random_stratified_70_10_20,train,2389,745,1644,31.18,69.98
2,random_stratified_70_10_20,validation,342,107,235,31.29,10.02


In [19]:
print('Random split label distribution:')
display(random_split.groupby(['split', 'label_str']).size().unstack(fill_value=0))

print('K-fold label distribution:')
display(kfold_split.groupby(['fold', 'label_str']).size().unstack(fill_value=0))

print('Split summary preview:')
display(split_summary.head(30))

Random split label distribution:


label_str,Clickbait,Non-clickbait
split,,
test,213,470
train,745,1644
validation,107,235


K-fold label distribution:


label_str,Clickbait,Non-clickbait
fold,,
0,213,470
1,213,470
2,213,470
3,213,470
4,213,469


Split summary preview:


,protocol,split,n_total,n_clickbait,n_non_clickbait,clickbait_rate,share_of_dataset,heldout_source,heldout_category
0,random_stratified_70_10_20,test,683,213,470,31.19,20.01,NaN,NaN
1,random_stratified_70_10_20,train,2389,745,1644,31.18,69.98,NaN,NaN
2,random_stratified_70_10_20,validation,342,107,235,31.29,10.02,NaN,NaN
3,stratified_kfold_5,fold_0_validation,683,213,470,31.19,20.01,NaN,NaN
4,stratified_kfold_5,fold_1_validation,683,213,470,31.19,20.01,NaN,NaN
5,stratified_kfold_5,fold_2_validation,683,213,470,31.19,20.01,NaN,NaN
6,stratified_kfold_5,fold_3_validation,683,213,470,31.19,20.01,NaN,NaN
7,stratified_kfold_5,fold_4_validation,682,213,469,31.23,19.98,NaN,NaN
8,leave_one_source_out::24h,test,564,248,316,43.97,16.52,24h,NaN
9,leave_one_source_out::24h,train,2850,817,2033,28.67,83.48,24h,NaN
